# Part 3 (실습) — LCEL 파이프(|)로 블록 잇기

> **이 노트북의 목표**
> 프롬프트와 모델을 파이프 `|` 하나로 잇고, 체인 전체에 `.invoke / .stream / .batch`를 써봄. `RunnableParallel`로 한 입력에서 여러 결과를 동시에 뽑고, `RunnableLambda`로 출력을 후처리함.
>
> **사용 모델**: `gemini-2.5-flash` · **선행**: Part 0~2

## 0. 준비

In [ ]:
# ─── 패키지 설치 ───
# %pip install: 주피터 노트북 안에서 파이썬 패키지를 설치하는 매직 명령어
#   -q (quiet): 설치 로그 최소화 | -U (upgrade): 최신 버전으로 업그레이드
%pip install -q -U langchain langchain-google-genai

In [ ]:
# os: 파이썬 내장 모듈 — 환경 변수(GOOGLE_API_KEY 등)를 읽고 쓸 때 사용
import os
# getpass: 입력한 글자를 화면에 숨겨서 받는 함수 (비밀번호 입력처럼)
#   → API 키를 노출 없이 안전하게 입력받기 위해 사용
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API 키: ")
# ChatGoogleGenerativeAI: Google Gemini 모델을 LangChain에서 사용할 수 있게 감싸주는 클래스
#   → langchain-google-genai 패키지에서 제공
#   → .invoke() / .stream() / .batch() 등 LangChain 공통 메서드를 지원
from langchain_google_genai import ChatGoogleGenerativeAI
# ChatPromptTemplate: 채팅 모델용 메시지 템플릿을 만드는 클래스
#   → .from_messages([(역할, 템플릿), ...]): 역할별 메시지 리스트 생성
#   → 역할: "system"(시스템), "human"(사용자), "ai"(모델 응답)
from langchain_core.prompts import ChatPromptTemplate
# ─── Gemini 모델 인스턴스 생성 ───
# model: 사용할 Gemini 모델명 (예: "gemini-2.5-flash" — 빠르고 저렴한 기본 모델)
# temperature: 답변의 무작위성 조절 (0=일관된 답 / 1=창의적·다양한 답)
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)
print("준비 완료")

## 1. 첫 체인 — 프롬프트 | 모델

### 문법
`chain = prompt | model`
- 왼쪽 블록의 **출력**이 오른쪽 블록의 **입력**으로 흐름 (컨베이어 벨트)
- 만든 체인은 `.invoke(입력)`으로 실행

In [ ]:
prompt = ChatPromptTemplate.from_template("{product}의 홍보 문구를 한 줄 써줘")

chain = prompt | model          # ← 파이프로 연결

# .invoke(): 입력을 모델/체인에 보내고 결과를 받는 LangChain의 핵심 실행 메서드
#   → 모든 LangChain 부품(Runnable)이 이 메서드를 공유함
result = chain.invoke({"product": "베이지 니트"})
print(type(result).__name__, "→", result.content)

### 데이터가 흐르는 과정을 단계별로 확인
파이프가 내부에서 하는 일을 손으로 펼쳐 보면 이해가 명확해짐.

In [ ]:
step1 = prompt.invoke({"product": "베이지 니트"})   # 1) 프롬프트가 메시지 리스트 생성
print("1단계 (prompt 출력):", [m.content for m in step1.messages])

step2 = model.invoke(step1)                          # 2) 그 리스트가 모델 입력으로
print("2단계 (model 출력):", step2.content)

# → chain.invoke()는 위 두 단계를 자동으로 이어준 것

## 2. 체인 전체에 invoke / stream / batch

### 핵심
체인도 다시 Runnable이라, 개별 블록에 쓰던 실행 메서드가 **체인 통째로** 그대로 적용됨.

In [ ]:
# batch — 여러 입력 한꺼번에
# .batch(): 여러 입력을 한꺼번에 보내 동시에 처리 (공동구매 방식)
#   → 리스트로 여러 질문을 넘기면 결과도 리스트로 반환
results = chain.batch([{"product": "니트"}, {"product": "코트"}, {"product": "스카프"}])
for r in results:
    print("•", r.content.strip())

In [ ]:
# stream — 마지막(모델)부터 조각이 실시간으로 흘러나옴
print("스트리밍:\n")
# .stream(): 응답을 한 번에 받지 않고 토큰 단위로 실시간 스트리밍
#   → for chunk in llm.stream(...): 형태로 사용, 타이핑 효과 구현
for chunk in chain.stream({"product": "트렌치 코트"}):
    print(chunk.content, end="", flush=True)
print()

> 📌 어디서 스트리밍이 시작되는지 우리가 챙길 필요 없음 — Runnable 규격이 알아서 전파함.

## 3. 출력 후처리 — 파이프에 함수 끼우기

모델 출력은 `AIMessage` 객체임. 본문 문자열만 깔끔히 받고 싶으면 파이프 끝에 함수를 끼움. (Part 4의 출력 파서가 이걸 정식화한 것)

In [ ]:
# 파이프 안의 람다는 자동으로 RunnableLambda로 변환됨
chain_clean = prompt | model | (lambda msg: msg.content.strip())

out = chain_clean.invoke({"product": "캐시미어 머플러"})
print(type(out).__name__, "→", out)   # 이제 str

## 4. RunnableParallel · RunnablePassthrough — 흐름 다루기

### 개념
- `RunnablePassthrough` : 입력을 가공 없이 그대로 통과 (복사기 '원본 통과')
- `RunnableParallel` : 여러 블록을 동시에 실행해 결과를 딕셔너리로 모음 (분배기)

In [ ]:
# ─── LCEL(LangChain Expression Language) Runnable 부품 ───
# RunnableParallel: 여러 작업을 동시에(병렬로) 실행하는 부품
#   → 입력을 여러 체인에 동시에 보내고 결과를 딕셔너리로 합침
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

parallel = RunnableParallel(
    original=RunnablePassthrough(),     # 입력 그대로
    length=lambda x: len(x),            # 입력 길이
    upper=lambda x: x.upper(),          # 대문자
)
print(parallel.invoke("beige knit"))
# → {'original': 'beige knit', 'length': 10, 'upper': 'BEIGE KNIT'}

### 실전 맛보기: 한 상품에서 두 종류 문구를 동시에
같은 상품을 받아 '감성 문구'와 '실용 문구'를 병렬로 생성함.

In [ ]:
emotional = ChatPromptTemplate.from_template("{product}의 감성적인 홍보 문구 한 줄") | model | (lambda m: m.content.strip())
practical = ChatPromptTemplate.from_template("{product}의 실용적 장점 한 줄") | model | (lambda m: m.content.strip())

both = RunnableParallel(감성=emotional, 실용=practical)
res = both.invoke({"product": "울 코트"})
print("감성:", res["감성"])
print("실용:", res["실용"])

> 📌 이 "입력을 갈래로 나눠 동시에 처리" 패턴이 RAG 체인(질문은 그대로 + 동시에 문서 검색)의 뼈대임 (Part 8).

## 5. RunnableLambda — 내 함수를 블록으로

복잡한 가공은 함수로 빼서 `RunnableLambda`로 명시하면 가독성이 좋아짐.

In [ ]:
# RunnableLambda: 일반 파이썬 함수를 LCEL 파이프라인 부품으로 변환
#   → 어떤 함수든 | (파이프)로 연결할 수 있게 감싸줌
from langchain_core.runnables import RunnableLambda

def to_hashtags(text: str) -> str:
    """문구를 해시태그 3개로 변환"""
    words = [w.strip(".,!") for w in text.split() if len(w) > 1][:3]
    return " ".join("#" + w for w in words)

chain_tag = (
    ChatPromptTemplate.from_template("{product}의 홍보 문구 한 줄")
    | model
    | (lambda m: m.content)
    | RunnableLambda(to_hashtags)      # 내 함수를 블록으로 끼움
)
print(chain_tag.invoke({"product": "린넨 셔츠"}))

## ✏️ 미니 실습

**과제**: 다음 체인을 만들기.
1. `ChatPromptTemplate`: "{topic}을 한 문장으로 설명해줘"
2. `model` 연결
3. 끝에 람다로 `.content`만 꺼내고 앞에 "💡 "를 붙이기
4. `topic="LCEL"`로 호출

In [ ]:
# TODO: 직접 작성
# chain = (...) | model | (lambda m: ...)
# print(chain.invoke({"topic": "LCEL"}))

<details><summary>👉 정답</summary>

```python
chain = (
    ChatPromptTemplate.from_template("{topic}을 한 문장으로 설명해줘")
    | model
    | (lambda m: "💡 " + m.content.strip())
)
print(chain.invoke({"topic": "LCEL"}))
```
</details>

## 정리

| 절 | 한 일 | 핵심 |
|---|---|---|
| 1 | 파이프 체인 | `prompt | model`, 출력→입력 흐름 |
| 2 | 체인 실행 | `.invoke / .stream / .batch` 통째 적용 |
| 3 | 후처리 람다 | 파이프 속 함수 = 자동 RunnableLambda |
| 4 | 병렬·통과 | `RunnableParallel` / `RunnablePassthrough` |
| 5 | 함수 블록 | `RunnableLambda(내함수)` |

### 3줄 요약
1. LCEL의 심장은 파이프 `|` — 블록을 레고처럼 잇고, 체인도 다시 Runnable임.
2. 실행 메서드(invoke/stream/batch)는 체인 전체에 그대로 적용됨.
3. Parallel·Passthrough·Lambda로 흐름을 나누고 끼워 넣음.

### ▶ 다음 (Part 4)
모델 출력(AIMessage)을 깔끔한 문자열·구조화 데이터로 받는 **출력 파서**를 배움. `프롬프트 | 모델 | 파서` 3단 체인이 완성됨.